[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PacktPublishing/Modern-Computer-Vision-with-PyTorch-2E/blob/main/Chapter10/action_recognition.ipynb)

In [3]:
%%capture
# This will take approx 3 min
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install -U "openmim==0.3.9"
!mim install -U "mmengine==0.10.4"
!mim install "mmcv==2.2.0"
!git clone https://github.com/sizhky/mmaction2.git -b main
%cd mmaction2
%pip install -e .
%pip install -r requirements/optional.txt
%pip install "timm==0.9.16"
%pip install "torch-snippets==0.528" lovely-tensors
!pip uninstall -y mmcv mmcv-full mmengine mmaction2
!pip install -U pip setuptools wheel
!pip install mmengine
!pip install mmcv-lite
!pip install -e .
!pip uninstall -y mmcv mmcv-lite

!pip install "mmcv-lite>=2.1.0,<2.2.0"

In [4]:
import mmengine
import mmcv

print("MMEngine:", mmengine.__version__)
print("MMCV:", mmcv.__version__)

MMEngine: 0.10.7
MMCV: 2.1.0


In [5]:
%cd /content

!rm -rf mmaction2
!git clone https://github.com/open-mmlab/mmaction2.git

%cd /content/mmaction2


/content
Cloning into 'mmaction2'...
remote: Enumerating objects: 22868, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 22868 (delta 0), reused 0 (delta 0), pack-reused 22866 (from 2)
Receiving objects: 100% (22868/22868), 69.63 MiB | 6.18 MiB/s, done.
Resolving deltas: 100% (16106/16106), done.
/content/mmaction2


In [6]:
import mmaction
print("MMAction2:", mmaction.__version__)

MMAction2: 1.2.0


In [7]:
# Check Pytorch installation
# %reload_ext autoreload
# %autoreload 2
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())
# 2.2.1+cu121 True

# Check MMAction2 installation
import mmaction
print(mmaction.__version__)
# 1.2.0

# Check MMCV installation
try:
    from mmcv.ops import get_compiling_cuda_version, get_compiler_version
    print(get_compiling_cuda_version())
    print(get_compiler_version())
except Exception as e:
    print("Skipping MMCV ops check:", e)

# Check MMEngine installation
from mmengine.utils.dl_utils import collect_env
print(collect_env())
_ = """
OrderedDict([
    ('sys.platform', 'linux'),
    ('Python', '3.10.12 (main, Nov 20 2023, 15:14:05) [GCC 11.4.0]'),
    ('CUDA available', True),
    ('MUSA available', False),
    ('numpy_random_seed', 2147483648),
    ('GPU 0', 'Tesla T4'),
    ('CUDA_HOME', '/usr/local/cuda'),
    ('NVCC', 'Cuda compilation tools, release 12.2, V12.2.140'),
    ('GCC', 'x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0'),
    ('PyTorch', '2.2.1+cu121'),
    ('PyTorch compiling details', '''
        PyTorch built with:
            - GCC 9.3
            - C++ Version: 201703
            - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
            - Intel(R) MKL-DNN v3.3.2 (Git Hash 2dc95a2ad0841e29db8b22fbccaf3e5da7992b01)
            - OpenMP 201511 (a.k.a. OpenMP 4.5)
            - LAPACK is enabled (usually provided by MKL)
            - NNPACK is enabled
            - CPU capability usage: AVX2
            - CUDA Runtime 12.1
            - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_70,code=sm_70;-gencode;arch=compute_75,code=sm_75;-gencode;arch=compute_80,code=sm_80;-gencode;arch=compute_86,code=sm_86;-gencode;arch=compute_90,code=sm_90
            - CuDNN 8.9.2
            - Magma 2.6.1
            - Build settings: BLAS_INFO=mkl, BUILD_TYPE=Release, CUDA_VERSION=12.1, CUDNN_VERSION=8.9.2, CXX_COMPILER=/opt/rh/devtoolset-9/root/usr/bin/c++, CXX_FLAGS= -D_GLIBCXX_USE_CXX11_ABI=0 -fabi-version=11 -fvisibility-inlines-hidden -DUSE_PTHREADPOOL -DNDEBUG -DUSE_KINETO -DLIBKINETO_NOROCTRACER -DUSE_FBGEMM -DUSE_QNNPACK -DUSE_PYTORCH_QNNPACK -DUSE_XNNPACK -DSYMBOLICATE_MOBILE_DEBUG_HANDLE -O2 -fPIC -Wall -Wextra -Werror=return-type -Werror=non-virtual-dtor -Werror=bool-operation -Wnarrowing -Wno-missing-field-initializers -Wno-type-limits -Wno-array-bounds -Wno-unknown-pragmas -Wno-unused-parameter -Wno-unused-function -Wno-unused-result -Wno-strict-overflow -Wno-strict-aliasing -Wno-stringop-overflow -Wsuggest-override -Wno-psabi -Wno-error=pedantic -Wno-error=old-style-cast -Wno-missing-braces -fdiagnostics-color=always -faligned-new -Wno-unused-but-set-variable -Wno-maybe-uninitialized -fno-math-errno -fno-trapping-math -Werror=format -Wno-stringop-overflow, LAPACK_INFO=mkl, PERF_WITH_AVX=1, PERF_WITH_AVX2=1, PERF_WITH_AVX512=1, TORCH_VERSION=2.2.1, USE_CUDA=ON, USE_CUDNN=ON, USE_EXCEPTION_PTR=1, USE_GFLAGS=OFF, USE_GLOG=OFF, USE_MKL=ON, USE_MKLDNN=ON, USE_MPI=OFF, USE_NCCL=1, USE_NNPACK=ON, USE_OPENMP=ON, USE_ROCM=OFF, USE_ROCM_KERNEL_ASSERT=OFF,
        '''
    )
    ('TorchVision', '0.17.1+cu121'),
    ('OpenCV', '4.8.0'),
    ('MMEngine', '0.10.4')
])
"""

2.11.0+cu128 True
1.2.0
Skipping MMCV ops check: No module named 'mmcv._ext'
OrderedDict({'sys.platform': 'linux', 'Python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'CUDA available': True, 'MUSA available': False, 'numpy_random_seed': np.uint32(2147483648), 'GPU 0': 'Tesla T4', 'CUDA_HOME': '/usr/local/cuda', 'NVCC': 'Cuda compilation tools, release 12.8, V12.8.93', 'GCC': 'x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0', 'PyTorch': '2.11.0+cu128', 'PyTorch compiling details': 'PyTorch built with:\n  - GCC 13.3\n  - C++ Version: 201703\n  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications\n  - Intel(R) MKL-DNN v3.10.2 (Git Hash f1d471933dc852f956fd05389f9313c7148783d5)\n  - OpenMP 201511 (a.k.a. OpenMP 4.5)\n  - LAPACK is enabled (usually provided by MKL)\n  - NNPACK is enabled\n  - CPU capability usage: AVX512\n  - CUDA Runtime 12.8\n  - NVCC architecture flags: -gencode;arch=compute_75,code

In [8]:
import sys, torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu128
CUDA: 12.8


In [9]:
# import sys
# print(sys.executable)

# import torch
# print(torch.__version__)

# try:
#     import mmcv
#     print("MMCV:", mmcv.__version__)
# except Exception as e:
#     print("MMCV error:", e)

# try:
#     import mmengine
#     print("MMEngine:", mmengine.__version__)
# except Exception as e:
#     print("MMEngine error:", e)

## Perform inference with a MMAction2 recognizer
MMAction2 already provides high level APIs to do inference and training.

In [10]:
!mkdir checkpoints
!wget -c https://download.openmmlab.com/mmaction/recognition/tsn/tsn_r50_1x1x3_100e_kinetics400_rgb/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth \
      -O checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth

--2026-06-10 15:38:57--  https://download.openmmlab.com/mmaction/recognition/tsn/tsn_r50_1x1x3_100e_kinetics400_rgb/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth
Resolving download.openmmlab.com (download.openmmlab.com)... 155.102.54.132, 155.102.54.133, 155.102.54.134, ...
Connecting to download.openmmlab.com (download.openmmlab.com)|155.102.54.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 97579339 (93M) [application/octet-stream]
Saving to: ‘checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth’

checkpoints/tsn_r50 100%[===================>]  93.06M  13.4MB/s    in 7.4s    

2026-06-10 15:39:04 (12.5 MB/s) - ‘checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth’ saved [97579339/97579339]



In [11]:
from torch_snippets import *
from builtins import print

from mmaction.apis import inference_recognizer, init_recognizer
from mmengine import Config


# Choose to use a config and initialize the recognizer
config = 'configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py'
config = Config.fromfile(config)
# Setup a checkpoint file to load
checkpoint = 'checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth'
# Initialize the recognizer
model = init_recognizer(config, checkpoint, device='cuda:0')

/usr/local/lib/python3.12/dist-packages/mmcv/cnn/bricks/transformer.py:33: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv`` rather than ``mmcv-lite`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '
/content/mmaction2/mmaction/evaluation/metrics/multimodal_metric.py:27: SyntaxWarning: invalid escape sequence '\d'
  commaStrip = re.compile('(\d)(,)(\d)')  # noqa: W605
/content/mmaction2/mmaction/evaluation/metrics/multimodal_metric.py:28: SyntaxWarning: invalid escape sequence '\d'
  periodStrip = re.compile('(?!<=\d)(\.)(?!\d)')  # noqa: W605


ImportError: cannot import name 'apply_chunking_to_forward' from 'transformers.modeling_utils' (/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py)

In [ ]:
model

In [ ]:
# Use the recognizer to do inference
from operator import itemgetter

def predict():
    video = 'demo/demo.mp4'
    label = 'tools/data/kinetics/label_map_k400.txt'
    results = inference_recognizer(model, video)

    pred_scores = results.pred_score.tolist()
    score_tuples = tuple(zip(range(len(pred_scores)), pred_scores))
    score_sorted = sorted(score_tuples, key=itemgetter(1), reverse=True)
    top5_label = score_sorted[:5]

    labels = open(label).readlines()
    labels = [x.strip() for x in labels]
    results = [(labels[k[0]], k[1]) for k in top5_label]
    return results

from torch_snippets.trainer.hooks import print_module_ios_for
with print_module_ios_for(model, required_children=['backbone','cls_head.consensus', 'cls_head.avg_pool', 'cls_head.dropout']) as _:
    results = predict()


In [ ]:
print('The top-5 labels with corresponding scores are:')
for result in results:
    print(f'{result[0]}: ', result[1])

## Train a recognizer on customized dataset

To train a new recognizer, there are usually three things to do:
1. Support a new dataset
2. Modify the config
3. Train a new recognizer

### Support a new dataset

In this tutorial, we gives an example to convert the data into the format of existing datasets. Other methods and more advanced usages can be found in the [doc](/docs/tutorials/new_dataset.md)

Firstly, let's download a tiny dataset obtained from [Kinetics-400](https://deepmind.com/research/open-source/open-source-datasets/kinetics/). We select 30 videos with their labels as train dataset and 10 videos with their labels as test dataset.

In [ ]:
# download, decompress the data
!rm kinetics400_tiny.zip*
!rm -rf kinetics400_tiny
!wget https://download.openmmlab.com/mmaction/kinetics400_tiny.zip
!unzip kinetics400_tiny.zip > /dev/null

In [ ]:
Glob('kinetics400_tiny/*/*')

In [ ]:
# After downloading the data, we need to check the annotation format
!cat kinetics400_tiny/kinetics_tiny_train_video.txt

According to the format defined in [`VideoDataset`](./datasets/video_dataset.py), each line indicates a sample video with the filepath and label, which are split with a whitespace.

### Modify the config

In the next step, we need to modify the config for the training.
To accelerate the process, we finetune a recognizer using a pre-trained recognizer.

In [ ]:
from mmengine import Config
cfg = Config.fromfile('./configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py')

Given a config that trains a TSN model on kinetics400-full dataset, we need to modify some values to use it for training TSN on Kinetics400-tiny dataset.


In [ ]:
from mmengine.runner import set_random_seed

# Modify dataset type and path
cfg.data_root = 'kinetics400_tiny/train/'
cfg.data_root_val = 'kinetics400_tiny/val/'
cfg.ann_file_train = 'kinetics400_tiny/kinetics_tiny_train_video.txt'
cfg.ann_file_val = 'kinetics400_tiny/kinetics_tiny_val_video.txt'


cfg.test_dataloader.dataset.ann_file = 'kinetics400_tiny/kinetics_tiny_val_video.txt'
cfg.test_dataloader.dataset.data_prefix.video = 'kinetics400_tiny/val/'

cfg.train_dataloader.dataset.ann_file = 'kinetics400_tiny/kinetics_tiny_train_video.txt'
cfg.train_dataloader.dataset.data_prefix.video = 'kinetics400_tiny/train/'

cfg.val_dataloader.dataset.ann_file = 'kinetics400_tiny/kinetics_tiny_val_video.txt'
cfg.val_dataloader.dataset.data_prefix.video  = 'kinetics400_tiny/val/'


# Modify num classes of the model in cls_head
cfg.model.cls_head.num_classes = 2
# We can use the pre-trained TSN model
cfg.load_from = './checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth'

# Set up working dir to save files and logs.
cfg.work_dir = './output'

# The original learning rate (LR) is set for 8-GPU training.
# We divide it by 8 since we only use one GPU.
cfg.train_dataloader.batch_size = cfg.train_dataloader.batch_size // 16
cfg.val_dataloader.batch_size = cfg.val_dataloader.batch_size // 16
cfg.optim_wrapper.optimizer.lr = cfg.optim_wrapper.optimizer.lr / 8 / 16
cfg.train_cfg.max_epochs = 10

cfg.train_dataloader.num_workers = 2
cfg.val_dataloader.num_workers = 2
cfg.test_dataloader.num_workers = 2

# We can initialize the logger for training and have a look
# at the final config used for training
print(f'Config:\n{cfg.pretty_text}')


### Train a new recognizer

Finally, lets initialize the dataset and recognizer, then train a new recognizer!

In [ ]:
import os.path as osp
import mmengine
from mmengine.runner import Runner

# Create work_dir
mmengine.mkdir_or_exist(osp.abspath(cfg.work_dir))

# build the runner from config
runner = Runner.from_cfg(cfg)

# start training
trained_model = runner.train()

In [ ]:
runner.test()